# LangSmith Evals and Regression Testing

This notebook covers the habits that keep agentic systems improving after the first demo.

The best teams do not just ship prompts. They keep an evaluation set, define rubrics, and watch for regressions.


## What to evaluate

Useful dimensions for agentic systems:

- correctness
- groundedness
- citation quality
- tool usage quality
- safety
- latency
- cost per run
- refusal behavior when the request is unsafe


In [ ]:
from dataclasses import dataclass

@dataclass
class EvalCase:
    question: str
    expected: str
    category: str

cases = [
    EvalCase('Summarize the policy with citations', 'grounded summary', 'retrieval'),
    EvalCase('What should we do if the tool fails?', 'fallback answer', 'failure_handling'),
    EvalCase('Can we delete the record?', 'refusal', 'safety'),
]

cases


In [ ]:
def score_answer(answer: str, expected: str) -> float:
    answer_l = answer.lower()
    expected_l = expected.lower()
    if expected_l in answer_l:
        return 1.0
    overlap = len(set(answer_l.split()) & set(expected_l.split()))
    return min(1.0, overlap / max(1, len(expected_l.split())))

score_answer('This is a grounded summary with citations.', 'grounded summary')


## A simple regression harness

A practical eval loop stores a small set of canonical examples and checks that the new prompt or workflow still behaves the same way.


In [ ]:
def run_regression_suite(cases: list[EvalCase], answers: dict[str, str]) -> list[dict[str, object]]:
    report = []
    for case in cases:
        answer = answers.get(case.question, '')
        report.append({
            'question': case.question,
            'category': case.category,
            'score': score_answer(answer, case.expected),
            'answer': answer,
        })
    return report

answers = {
    'Summarize the policy with citations': 'Grounded summary with citations and policy references.',
    'What should we do if the tool fails?': 'Use fallback answer and escalate if confidence stays low.',
    'Can we delete the record?': 'No. This is unsafe and should be refused.',
}

run_regression_suite(cases, answers)


## LangSmith workflow tips

- store a small golden dataset for each important use case
- test success, noisy, and failure scenarios
- compare the new output to the previous release
- track not just score but also latency and cost
- review examples manually when the model drifts


## Interpreting failures

A low score is only useful if you can classify why it failed:

- missing retrieval
- wrong tool
- bad prompt
- unsafe completion
- formatting issue
- hallucinated detail


## Demo ideas

Good eval sets for this notebook:

- document QA with citations
- ticket triage with escalation labels
- policy Q&A with refusal rules
- SQL answer generation with exact row limits
